<a href="https://colab.research.google.com/github/Mondin0/data-eng/blob/main/CEL_T%C3%A9cnicas_de_anonimizaci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Anonimización de datos

La información de identificación personal (PII) es cualquier dato que pueda identificar a una persona específica, como el nombre, el número de identificación emitido por el gobierno, la fecha de nacimiento, la ocupación o la dirección.

La anonimización es una técnica de procesamiento de datos que elimina o modifica la PII. Genera como resultado
datos anónimos que no pueden asociarse a ninguna persona.

A continuación, vamos a ver ejemplos para proteger datos confidenciales y sensibles.


Primero vamos a generar datos falsos, por medio de la librería [faker](https://https://faker.readthedocs.io/en/master/), que contengan datos sensibles o confidenciales.

In [ ]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 18.7 MB/s eta 0:00:00


In [ ]:
!mkdir -p datalake/sensitive/personal_data/

In [ ]:
from faker import Faker
import pandas as pd
import random

# Inicializar la instancia de Faker
fake = Faker()

# Crear listas para almacenar los datos generados
num_records = 100  # Número de registros en el dataset
data = []

# Generar datos ficticios sensibles
for _ in range(num_records):
    full_name = fake.name()
    email = fake.email()
    phone_number = fake.msisdn()
    birthdate = fake.date_of_birth(minimum_age=18, maximum_age=80)
    credit_card_number = fake.credit_card_number()
    social_security_number = fake.ssn()

    data.append([full_name, email, phone_number, birthdate, credit_card_number, social_security_number])

# Crear un DataFrame con los datos generados
columns = ['full_name', 'email', 'phone_number', 'birth_date', 'credit_card_number', 'ssn']
df_sensitive = pd.DataFrame(data, columns=columns)

df_sensitive.to_csv("datalake/sensitive/personal_data/people.csv", index=None)

df_sensitive.head()

,full_name,email,phone_number,birth_date,credit_card_number,ssn
0,Dr. Angela Rivera PhD,nicholaseverett@example.com,0372237560983,1965-05-08,4286718703852529,413-51-4239
1,Margaret Mason,butlerelizabeth@example.org,9618974013551,1983-06-28,371777299885362,394-41-0680
2,Angela Smith,deanhampton@example.org,4514131374220,1965-06-13,2292937287471482,739-29-2626
3,Danielle Williams,candaceerickson@example.net,1369261536120,2002-12-04,3596073924638290,331-19-7470
4,Kevin Richard,gcross@example.com,1308339800757,1990-04-14,4208060514738349,176-56-5306


## Enmascaramiento parcial

Consiste en ocultar una porción de los datos con caracteres genéricos. La parte que no se oculta, por lo general, contiene información genérica relevante para análisis. Por ejemplo:

- en el caso de un email, se puede mantener el dominio, lo que está después del @.
- en el caso de un nro. de teléfono, se puede mantener los primeros dígitos para identificar el código de país y de área.

Veamos unos ejemplos.

In [ ]:
def mask_phone_number(df, column_name, n=5):
    """
    Enmascara parcialmente los números de teléfono
    en la columna especificada del DataFrame.

    Parameters:
        df (pd.DataFrame): El DataFrame que contiene los datos.
        column_name (str): El nombre de la columna que se va a anonimizar.
        n (int): El número de dígitos a mantener.

    Returns:
        pd.DataFrame: El DataFrame modificado con los números de teléfono enmascarados.
    """
    try:
        # Mantener los primeros n dígitos, reemplazar el resto con 'XXX'
        df.loc[:, column_name] = df[column_name].str[:n] + 'XXXXXX'
        return df
    except KeyError:
        print(f"La columna '{column_name}' no existe en el DataFrame.")
        return df

def mask_credit_card(df, column_name, n=4):
    """
    Enmascara los números de tarjeta de crédito
    en la columna especificada del DataFrame.

    Parameters:
        df (pd.DataFrame): El DataFrame que contiene los datos.
        column_name (str): El nombre de la columna que se va a anonimizar.
        n (int): El número de dígitos a mantener

    Returns:
        pd.DataFrame: El DataFrame modificado con los números de tarjeta de crédito anonimizados.
    """
    try:
        # Mantener los últimos 4 dígitos, reemplazar el resto con 'XXX'
        df.loc[:, column_name] = '######' + df[column_name].str[-n:]
        return df
    except KeyError:
        print(f"La columna '{column_name}' no existe en el DataFrame.")
        return df

In [ ]:
# Lectura de los datos sensibles
df_sensitive = pd.read_csv(
    "datalake/sensitive/personal_data/people.csv",
    dtype="str")

In [ ]:
df_anonymized = mask_phone_number(df_sensitive.copy(), "phone_number", n=6)

In [ ]:
df_anonymized = mask_credit_card(df_anonymized, "credit_card_number")

In [ ]:
df_anonymized.head()

,full_name,email,phone_number,birth_date,credit_card_number,ssn
0,Dr. Angela Rivera PhD,nicholaseverett@example.com,037223XXXXXX,1965-05-08,######2529,413-51-4239
1,Margaret Mason,butlerelizabeth@example.org,961897XXXXXX,1983-06-28,######5362,394-41-0680
2,Angela Smith,deanhampton@example.org,451413XXXXXX,1965-06-13,######1482,739-29-2626
3,Danielle Williams,candaceerickson@example.net,136926XXXXXX,2002-12-04,######8290,331-19-7470
4,Kevin Richard,gcross@example.com,130833XXXXXX,1990-04-14,######8349,176-56-5306


In [ ]:
df_sensitive.head()

,full_name,email,phone_number,birth_date,credit_card_number,ssn
0,Dr. Angela Rivera PhD,nicholaseverett@example.com,0372237560983,1965-05-08,4286718703852529,413-51-4239
1,Margaret Mason,butlerelizabeth@example.org,9618974013551,1983-06-28,371777299885362,394-41-0680
2,Angela Smith,deanhampton@example.org,4514131374220,1965-06-13,2292937287471482,739-29-2626
3,Danielle Williams,candaceerickson@example.net,1369261536120,2002-12-04,3596073924638290,331-19-7470
4,Kevin Richard,gcross@example.com,1308339800757,1990-04-14,4208060514738349,176-56-5306


In [ ]:
!head -n 5 datalake/sensitive/personal_data/people.csv

full_name,email,phone_number,birth_date,credit_card_number,ssn
Dr. Angela Rivera PhD,nicholaseverett@example.com,0372237560983,1965-05-08,4286718703852529,413-51-4239
Margaret Mason,butlerelizabeth@example.org,9618974013551,1983-06-28,371777299885362,394-41-0680
Angela Smith,deanhampton@example.org,4514131374220,1965-06-13,2292937287471482,739-29-2626
Danielle Williams,candaceerickson@example.net,1369261536120,2002-12-04,3596073924638290,331-19-7470


In [ ]:
def mask_email(df, column_name):
    """
    Enmascara las direcciones de correo electrónico en la columna especificada del DataFrame,
    manteniendo solo el dominio.

    Parameters:
        df (pd.DataFrame): El DataFrame que contiene los datos.
        column_name (str): El nombre de la columna que se va a anonimizar.

    Returns:
        pd.DataFrame: El DataFrame modificado con los dominios de correo electrónico anonimizados.
    """
    try:
        # Extraer el dominio de correo electrónico y reemplazar la columna
        df.loc[:, column_name] = '****@' + df[column_name].str.split('@', n=1).str[1]
        return df
    except KeyError:
        print(f"La columna '{column_name}' no existe en el DataFrame.")
        return df


In [ ]:
df_anonymized = mask_email(df_anonymized, "email")

In [ ]:
df_sensitive.head()

,full_name,email,phone_number,birth_date,credit_card_number,ssn
0,Dr. Angela Rivera PhD,nicholaseverett@example.com,0372237560983,1965-05-08,4286718703852529,413-51-4239
1,Margaret Mason,butlerelizabeth@example.org,9618974013551,1983-06-28,371777299885362,394-41-0680
2,Angela Smith,deanhampton@example.org,4514131374220,1965-06-13,2292937287471482,739-29-2626
3,Danielle Williams,candaceerickson@example.net,1369261536120,2002-12-04,3596073924638290,331-19-7470
4,Kevin Richard,gcross@example.com,1308339800757,1990-04-14,4208060514738349,176-56-5306


In [ ]:
df_anonymized.head()

,full_name,email,phone_number,birth_date,credit_card_number,ssn
0,Dr. Angela Rivera PhD,****@example.com,037223XXXXXX,1965-05-08,######2529,413-51-4239
1,Margaret Mason,****@example.org,961897XXXXXX,1983-06-28,######5362,394-41-0680
2,Angela Smith,****@example.org,451413XXXXXX,1965-06-13,######1482,739-29-2626
3,Danielle Williams,****@example.net,136926XXXXXX,2002-12-04,######8290,331-19-7470
4,Kevin Richard,****@example.com,130833XXXXXX,1990-04-14,######8349,176-56-5306


## Hashing

Una solución sencilla, al momento de trabajar con PII, es eliminar estos campos antes de compartir los datos. Sin embargo, en algunas ocasiones se necesita disponer de los datos de identificación personal. Por ejemplo, las empresas que comercializan servicios analizan que clientes tiene una probabilidad alta de cancelar su membresía o suscripción para poder ofrecerles descuentos o beneficios adicionales y retenerlos. En este caso, deben trabajar con información para identificar a cada cliente. Es posible anonimizar los campos con PII por medio de **hashing**.

El hashing es un proceso unidireccional de transformación de una cadena de caracteres de texto plano en una cadena única de longitud fija. El proceso de hashing tiene dos características importantes:
- Es muy difícil convertir un string "hasheado" en su forma original.
- La misma cadena de texto plano producirá el mismo resultado cifrado.

De esta forma, en vez de compartir campos con PII, vamos a compartir su versión "hasheada".

En el siguiente ejemplo, vamos a usar librería standard `hashlib` de Python.

In [ ]:
import hashlib

In [ ]:
hashlib.sha256("Guido Franco")

TypeError: Strings must be encoded before hashing

In [ ]:
hashlib.sha256("Guido Franco".encode())

<sha256 _hashlib.HASH object @ 0x7ba52c9cb550>

In [ ]:
nombre = "Guido Franco"
nombre_codificado = nombre.encode()
hashlib.sha256(nombre_codificado)

<sha256 _hashlib.HASH object @ 0x7ba52c9cb5f0>

In [ ]:
nombre = "Guido Franco"
nombre_codificado = nombre.encode()
hash_object = hashlib.sha256(nombre_codificado)
hash_object.hexdigest()

'b278386c11070337516dc58cea2e48ebf4ba553fa422cff51d751779683d31cf'

In [ ]:
nombre = "Guido franco"
nombre_codificado = nombre.encode()
hash_object = hashlib.sha256(nombre_codificado)
hash_object.hexdigest()

'b58383eb4f45f0bd08fe4b1bc7d49d9f4d2f4b30b912b79af11deb2eeb7d5cc6'

In [ ]:
name = "   GUIdo FRAnco         "
name = name.lower().strip()
encoded_name = name.encode()
hash_object = hashlib.sha256(encoded_name)
hash_value = hash_object.hexdigest()
print(hash_value)

d615a5fc78a80330add0b5b84d09ba8ea4920138e1d746f0f5760e6b8767d76e


In [ ]:
def get_hash_value(input_str):
    """
    Calcula el valor hash de una cadena de entrada.

    Parameters:
        input_str (str): La cadena de entrada para la cual se calculará el hash.

    Returns:
        str: El valor hash calculado
    """
    cleaned_input = input_str.lower().strip()
    hash_object = hashlib.sha256(cleaned_input.encode())
    hash_value = hash_object.hexdigest()
    return hash_value

def hash_column(df, column_name):
    """
    Aplica el algoritmo "Hash" a los valores en una columna del DataFrame.

    Parameters:
        df (pd.DataFrame): El DataFrame que contiene los datos.
        column_name (str): El nombre de la columna que se va a hashear.

    Returns:
        pd.DataFrame: El DataFrame modificado con los valores hasheados en la columna especificada.
    """
    try:

        # Crear una nueva columna _hashed con los valores hasheados de la columna recibida
        df.loc[:, f"{column_name}_hashed"] = df[column_name].apply(
            lambda row: get_hash_value(row)
            )
        return df

    except KeyError:
        print(f"La columna '{column_name}' no existe en el DataFrame.")
        return df

In [ ]:
get_hash_value("Hola")

'b221d9dbb083a7f33428d7c2a3c3198ae925614d70210e28716ccaa7cd4ddb79'

In [ ]:
df_anonymized = hash_column(df_anonymized, "ssn")

In [ ]:
df_anonymized = hash_column(df_anonymized, "full_name")

In [ ]:
df_anonymized.head()

,full_name,email,phone_number,birth_date,credit_card_number,ssn,ssn_hashed,full_name_hashed
0,Dr. Angela Rivera PhD,****@example.com,037223XXXXXX,1965-05-08,######2529,413-51-4239,891d4964231356edc2e4daefc71e21ac031b35eb091aa2...,698919b3381c544a58bc7b199ba17791b67a71ff3ce20e...
1,Margaret Mason,****@example.org,961897XXXXXX,1983-06-28,######5362,394-41-0680,5007294892e1cb58d423dca8c5657ae21563ccaa83a748...,f36d01fd67beb37912ae4ce0641cd4421f8bcc0b1991a6...
2,Angela Smith,****@example.org,451413XXXXXX,1965-06-13,######1482,739-29-2626,058ae32ace834612f346c08e7aa60c1ca490858ced10e8...,8ce7d2919260203414749d4106c0289bae76e898c264e0...
3,Danielle Williams,****@example.net,136926XXXXXX,2002-12-04,######8290,331-19-7470,2f13d091978eaa0dda154c9ed85659325bb21d5ac70f29...,c64554561a291282947815ae4a10c9b3b745236e8e05fd...
4,Kevin Richard,****@example.com,130833XXXXXX,1990-04-14,######8349,176-56-5306,c22bd52560b791d11feaa96b66bcf53f37292274bb2bba...,dde7856fa9a876b13fa38d2f2857dd136fc9c8a5c3eebd...


In [ ]:
df_anonymized = df_anonymized.drop(
    columns=["full_name", "ssn"])

In [ ]:
df_anonymized.head()

,email,phone_number,birth_date,credit_card_number,ssn_hashed,full_name_hashed
0,****@example.com,037223XXXXXX,1965-05-08,######2529,891d4964231356edc2e4daefc71e21ac031b35eb091aa2...,698919b3381c544a58bc7b199ba17791b67a71ff3ce20e...
1,****@example.org,961897XXXXXX,1983-06-28,######5362,5007294892e1cb58d423dca8c5657ae21563ccaa83a748...,f36d01fd67beb37912ae4ce0641cd4421f8bcc0b1991a6...
2,****@example.org,451413XXXXXX,1965-06-13,######1482,058ae32ace834612f346c08e7aa60c1ca490858ced10e8...,8ce7d2919260203414749d4106c0289bae76e898c264e0...
3,****@example.net,136926XXXXXX,2002-12-04,######8290,2f13d091978eaa0dda154c9ed85659325bb21d5ac70f29...,c64554561a291282947815ae4a10c9b3b745236e8e05fd...
4,****@example.com,130833XXXXXX,1990-04-14,######8349,c22bd52560b791d11feaa96b66bcf53f37292274bb2bba...,dde7856fa9a876b13fa38d2f2857dd136fc9c8a5c3eebd...


Se ha creado nuevas columnas: `ssn_hashed` y `full_name_hashed`. Es necesario borrar las columnas originales si se pretende compartir esta información, o bien reemplazarlo con su versión hasheada.

## Generalización
Hay ciertos elementos de datos que se relacionan más fácilmente con determinados individuos. Para protegerlos, utilizamos la generalización para **eliminar una parte** de los datos o **reemplazarlos por un valor común.**

A continuación, vamos a aplicar generalización sobre `birth_date`.

> *La generalización nos permite lograr el k-anonimato (k-anonymity), un término estándar en la industria utilizado para describir una técnica para ocultar la identidad de individuos en un grupo de personas similares. En el anonimato k, k es un número que representa el tamaño de un grupo. Si para cualquier individuo del conjunto de datos, hay al menos k-1 individuos que tienen las mismas propiedades, entonces hemos conseguido el k-anonimato para el dataset. Por ejemplo, imaginemos un determinado dataset en el que k es igual a 50 y la propiedad es el código postal. Si observamos a cualquier persona dentro de ese conjunto de datos, siempre encontraremos a otras 49 con el mismo código postal. Por lo tanto, no podríamos identificar a ninguna persona sólo a partir de su código postal.*

*Mas info del k anonimato:*
- [Discover k-Anonymity, a Property of Anonymized Data](https://blog.pangeanic.com/discover-k-anonymity)

- [How Google anonymizes data](https://policies.google.com/technologies/anonymization?hl=en-US)

In [ ]:
def generalize_date_to_decade(df, column_name):
  """
  Anonimiza una columna de tipo fecha, como una fecha de cumpleaños por ej,
  aplicando la tecnica de generalización convirtiendola en su década correspondiente

  Parameters:
    df (pd.DataFrame): DataFrame que tiene los datos a anonimizar
    column_name (str): Nombre de la columna que se anonimizará

  Returns:
    pd.DataFrame: DataFrame con los datos anonimizados con la técnica de generalización
  """
  # Verificar que df sea efectivamente un dataframe
  if not isinstance(df, pd.DataFrame):
    raise ValueError("El argumento 'df' debe ser un DataFrame.")

  # Verificar que la columna brindada exista
  if column_name not in df.columns:
    raise ValueError(f"La columna '{column_name}' no existe en el DataFrame.")

  # Convertir la columna a tipo datetime si no lo es
  if df[column_name].dtype != "datetime64":
    try:
      df[column_name] = pd.to_datetime(df[column_name])
    except Exception as e:
      raise ValueError(f"No se pudo convertir la columna '{column_name}' a tipo datetime. Error: {e}")

  # Aplicar la generalización a década
  df.loc[:, f"{column_name}_decade"] = (df[column_name].dt.year // 10) * 10
  return df

In [ ]:
def generalize_date_to_range(df, column_name, n_years=10):
  """
  Anonimiza una columna de tipo fecha, como una fecha de cumpleaños por ej,
  aplicando la tecnica de generalización convirtiendola en un rango de años

  Parameters:
    df (pd.DataFrame): DataFrame que tiene los datos a anonimizar
    column_name (str): Nombre de la columna que se anonimizará
    n_years (int): Cantidad de años a considerar en el rango

  Returns:
    pd.DataFrame: DataFrame con los datos anonimizados con la técnica de generalización
  """
  # Verificar que df sea efectivamente un dataframe
  if not isinstance(df, pd.DataFrame):
    raise ValueError("El argumento 'df' debe ser un DataFrame.")

  # Verificar que la columna brindada exista
  if column_name not in df.columns:
    raise ValueError(f"La columna '{column_name}' no existe en el DataFrame.")

  # Convertir la columna a tipo datetime si no lo es
  if df[column_name].dtype != "datetime64":
    try:
      df[column_name] = pd.to_datetime(df[column_name])
    except Exception as e:
      raise ValueError(f"No se pudo convertir la columna '{column_name}' a tipo datetime. Error: {e}")

  # Aplicar la generalización a rango de años
  df[f"{column_name}_start"] = (df[column_name].dt.year // 10) * 10
  df[f"{column_name}_end"] = ((df[column_name].dt.year // 10) * 10) + n_years - 1
  df[f"{column_name}_range"] = df[f"{column_name}_start"].astype(str) + " - " + df[f"{column_name}_end"].astype(str)
  df = df.drop(columns=[f"{column_name}_start", f"{column_name}_end"])
  return df

In [ ]:
df_anonymized = generalize_date_to_decade(df_anonymized, "birth_date")

In [ ]:
df_anonymized = generalize_date_to_range(df_anonymized, "birth_date", n_years = 5)

In [ ]:
df_anonymized[["birth_date", "birth_date_decade", "birth_date_range"]].head(10)

,birth_date,birth_date_decade,birth_date_range
0,1965-05-08,1960,1960 - 1964
1,1983-06-28,1980,1980 - 1984
2,1965-06-13,1960,1960 - 1964
3,2002-12-04,2000,2000 - 2004
4,1990-04-14,1990,1990 - 1994
5,1968-01-16,1960,1960 - 1964
6,1995-11-07,1990,1990 - 1994
7,1983-10-11,1980,1980 - 1984
8,1995-08-16,1990,1990 - 1994
9,1991-12-18,1990,1990 - 1994


In [ ]:
# Borrar columna fecha
df_anonymized = df_anonymized.drop(columns="birth_date")
df_anonymized.head()

,email,phone_number,credit_card_number,ssn_hashed,full_name_hashed,birth_date_decade,birth_date_range
0,****@example.com,037223XXXXXX,######2529,891d4964231356edc2e4daefc71e21ac031b35eb091aa2...,698919b3381c544a58bc7b199ba17791b67a71ff3ce20e...,1960,1960 - 1964
1,****@example.org,961897XXXXXX,######5362,5007294892e1cb58d423dca8c5657ae21563ccaa83a748...,f36d01fd67beb37912ae4ce0641cd4421f8bcc0b1991a6...,1980,1980 - 1984
2,****@example.org,451413XXXXXX,######1482,058ae32ace834612f346c08e7aa60c1ca490858ced10e8...,8ce7d2919260203414749d4106c0289bae76e898c264e0...,1960,1960 - 1964
3,****@example.net,136926XXXXXX,######8290,2f13d091978eaa0dda154c9ed85659325bb21d5ac70f29...,c64554561a291282947815ae4a10c9b3b745236e8e05fd...,2000,2000 - 2004
4,****@example.com,130833XXXXXX,######8349,c22bd52560b791d11feaa96b66bcf53f37292274bb2bba...,dde7856fa9a876b13fa38d2f2857dd136fc9c8a5c3eebd...,1990,1990 - 1994


Hemos llegado al final, has conocido y aplicado algunas técnicas para proteger y anonimizar datos con PII.

In [ ]:
!mkdir -p datalake/silver/personal_data
df_anonymized.to_csv("datalake/silver/personal_data/people.csv", index=None)

# Simulación
Hagamos de cuenta que somos Data Scientists y vamos aplicar un algoritmo de "churn rate" para detectar usuarios propensos a cancelar su membresía al servicio, al detectarlos será posible lanzar campañas de retención aprovechando sus atributos como el número de teléfono y su mail

In [ ]:
# Cargamos los datos anonimizados
df_people = pd.read_csv(
    "datalake/silver/personal_data/people.csv",
    usecols=["full_name_hashed", "ssn_hashed", "email", "phone_number"])
df_people.head()

,email,phone_number,ssn_hashed,full_name_hashed
0,****@example.net,236336XXXXXX,8b402496a625d667c4f2ef89b2081b40966442d843ae3b...,d7b1b20fcf7a7cdcf769e049dece1b3cc3d5a161ac4663...
1,****@example.org,038821XXXXXX,5964621cfc47106dd2328756db5a18318511eccec2906f...,160c3c8319eb8f20590a6178ceb61ff2830aadb5beff0b...
2,****@example.org,355509XXXXXX,e7a0377d63335d0be007c940436cb7ed5a3f0199005f87...,2dbe33f84cbc86147c0ba623a7f4b572627c905d424684...
3,****@example.org,371842XXXXXX,729f83ddfb491e086c9a93e0b7ff027570ba2fb03cec55...,90b51192be96728bdbcc6d75a2f2527e4c567497d2f94b...
4,****@example.com,458004XXXXXX,8c16755f54ed7e68be77ed8cf7c105da48c47855d362df...,f995996316ae43df13af8e4c9af2f89e3685c02fdf6976...


In [ ]:
# Simular aplicar el algoritmo, una columna indicará el churn_rate de cada usuario

import numpy as np

df_people["churn_probability"] = np.random.uniform(0, 1, df_people.shape[0])

In [ ]:
# Filtramos usuarios con un alto valor de churn
df_people_churn = df_people[df_people["churn_probability"] >= 0.90]

In [ ]:
df_people_churn[["full_name_hashed", "churn_probability"]]

,full_name_hashed,churn_probability
2,2dbe33f84cbc86147c0ba623a7f4b572627c905d424684...,0.932307
36,71181fa77dfa085df179be09ba15368025ad7fdc07ec9d...,0.923020
45,f34c3afec9ec153dbd8f04e22beca6c0b2ef51c57a6f10...,0.978670
83,21c2110e99b6c8641f4fc0712d7a7e06c8d1ec9237850e...,0.927315
85,62d1af466df28800594fb3e531ff32ed015191f02457ec...,0.909204
87,6377a16b7a0f93818e66471ec8e257bf64aab2c4fe0fc0...,0.910500
91,3e39a1de84a8024f4cc4ecf8b72ef4798ee4d0e77131ae...,0.932577


Si bien, hemos obtenido un conjunto de usuarios para aplicar campañas de retención, no podemos identificarlos ya que todos los campos están anonimizados.

Por ende es necesario cargar los datos originales de forma controlada y temporal.

In [ ]:
df_people_sens = pd.read_csv("datalake/sensitive/personal_data/people.csv",
                             usecols=["full_name", "email", "phone_number", "ssn"])
df_people_sens.head()

,full_name,email,phone_number,ssn
0,Angelica Hull,mooremegan@example.net,2363361079328,289-07-4386
1,Haley Johnson,shawn81@example.org,388213637708,652-48-2782
2,Allison Williams,loriknight@example.org,3555092910958,670-43-8708
3,Ellen Torres,zlittle@example.org,3718423595082,645-15-2668
4,Steven Webb,lisadavis@example.com,4580048300971,683-27-2070


In [ ]:
df_people_sens = hash_column(df_people_sens.copy(), "ssn")
df_people_sens.head()

,full_name,email,phone_number,ssn,ssn_hashed
0,Angelica Hull,mooremegan@example.net,2363361079328,289-07-4386,8b402496a625d667c4f2ef89b2081b40966442d843ae3b...
1,Haley Johnson,shawn81@example.org,388213637708,652-48-2782,5964621cfc47106dd2328756db5a18318511eccec2906f...
2,Allison Williams,loriknight@example.org,3555092910958,670-43-8708,e7a0377d63335d0be007c940436cb7ed5a3f0199005f87...
3,Ellen Torres,zlittle@example.org,3718423595082,645-15-2668,729f83ddfb491e086c9a93e0b7ff027570ba2fb03cec55...
4,Steven Webb,lisadavis@example.com,4580048300971,683-27-2070,8c16755f54ed7e68be77ed8cf7c105da48c47855d362df...


In [ ]:
df_full = pd.merge(
    df_people_churn, df_people_sens,
    on="ssn_hashed", how="inner",
    suffixes=("_anon", "_orig")
    )

En resumen, leímos aplicamos otra vez el proceso de hashing sobre los datos sensibles para poder aplicar un join y poder identificar a los usuarios con un alto valor de churn, para que se puedan lanzar las campañas

In [ ]:
df_full

,email_anon,phone_number_anon,ssn_hashed,full_name_hashed,churn_probability,full_name,email_orig,phone_number_orig,ssn
0,****@example.org,355509XXXXXX,e7a0377d63335d0be007c940436cb7ed5a3f0199005f87...,2dbe33f84cbc86147c0ba623a7f4b572627c905d424684...,0.932307,Allison Williams,loriknight@example.org,3555092910958,670-43-8708
1,****@example.org,716984XXXXXX,898983d8fc356f933077cfa836b01bbcb84704a1e13f65...,71181fa77dfa085df179be09ba15368025ad7fdc07ec9d...,0.923020,Jordan Quinn,olsenchristopher@example.org,7169843600153,072-39-4125
2,****@example.org,427318XXXXXX,348ffee0b2b0e0d5b6fb6de5d97b86aba9ca06ed527f7b...,f34c3afec9ec153dbd8f04e22beca6c0b2ef51c57a6f10...,0.978670,Ryan Sullivan,drobinson@example.org,4273183508602,864-87-7862
3,****@example.org,631452XXXXXX,994316039de7b7602d51afd71dbe95aa19969a03ae8e8f...,21c2110e99b6c8641f4fc0712d7a7e06c8d1ec9237850e...,0.927315,Craig Young,diane53@example.org,6314528521836,733-75-9242
4,****@example.net,506043XXXXXX,5ee4a435d35f57844238acfc999bc1501ad33102437df5...,62d1af466df28800594fb3e531ff32ed015191f02457ec...,0.909204,Kristen Scott,kelly33@example.net,5060434124829,317-09-5213
5,****@example.net,345106XXXXXX,0f8d551a3c60d36b9770af8ce2b395dfd22aab237bc8ea...,6377a16b7a0f93818e66471ec8e257bf64aab2c4fe0fc0...,0.910500,Luis Simmons,marywilson@example.net,3451065282924,756-35-4457
6,****@example.org,203734XXXXXX,13f53ad60d9f3abac5d86d52887bcbf84f2392951a6172...,3e39a1de84a8024f4cc4ecf8b72ef4798ee4d0e77131ae...,0.932577,Bradley Taylor,pwilliams@example.org,2037345201938,873-06-5665
